In [1]:
import math
import sqlite3
import requests
import pandas as pd

headers = {
    'accept': 'application/json',
    'user-agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36',
    'referer': 'https://www.invitationhomes.com/search/houses-for-rent',
}

MARKETS = {
    'atlanta-georgia': (33.748995, -84.387982),
    'austin-texas': (30.266666, -97.73333),
    'charlotte-north-carolina': (35.521445, -79.905591),
    'chicago-illinois': (41.878114, -87.629798),
    'dallas-texas': (32.776664, -96.796988),
    'denver-colorado': (39.739236, -104.990251),
    'houston-texas': (29.760427, -95.369803),
    'jacksonville-florida': (30.332184, -81.655651),
    'las-vegas-nevada': (36.169941, -115.13983),
    'los-angeles-california': (34.052234, -118.243685),
    'miami-florida': (25.76168, -80.19179),
    'minneapolis-minnesota': (44.977753, -93.265011),
    'nashville-tennessee': (36.162664, -86.781602),
    'orlando-florida': (28.538335, -81.379236),
    'phoenix-arizona': (33.448377, -112.074037),
    'sacramento-california': (38.581572, -121.4944),
    'salt-lake-city-utah': (40.7608, -111.891),
    'san-antonio-texas': (29.424349, -98.491142),
    'seattle-washington': (47.606209, -122.332071),
    'tampa-florida': (27.950575, -82.457178),
}

print('Available markets:')
for m in sorted(MARKETS):
    print(f'  {m}')

Available markets:
  atlanta-georgia
  austin-texas
  charlotte-north-carolina
  chicago-illinois
  dallas-texas
  denver-colorado
  houston-texas
  jacksonville-florida
  las-vegas-nevada
  los-angeles-california
  miami-florida
  minneapolis-minnesota
  nashville-tennessee
  orlando-florida
  phoenix-arizona
  sacramento-california
  salt-lake-city-utah
  san-antonio-texas
  seattle-washington
  tampa-florida


In [2]:
metro_location = input('Enter metro location (e.g. houston-texas, atlanta-georgia, dallas-texas): ').strip().lower()

if metro_location not in MARKETS:
    raise ValueError(f'Unknown market: {metro_location}. See list above.')

lat, lng = MARKETS[metro_location]
bbox_pad = 0.65

records = []
LIMIT = 25

In [3]:
offset = 0
total = 999

while offset < total:
    url = (
        f'https://www.invitationhomes.com/property/api/geo-search'
        f'?baths_min=1&beds_min=1'
        f'&rent_min=0&rent_max=10000'
        f'&sqft_min=0&sqft_max=10000'
        f'&south={lat - bbox_pad}&west={lng - bbox_pad}'
        f'&north={lat + bbox_pad}&east={lng + bbox_pad}'
        f'&lat={lat}&long={lng}'
        f'&limit={LIMIT}&offset={offset}'
        f'&sort=distance&sort_direction=asc'
    )
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()
    if not data.get('properties'):
        break

    total = data['total']

    for item in data['properties']:
        addr = item.get('address', {})
        loc = item.get('map_location', {})
        records.append({
            'property_id': item.get('property_id', ''),
            'street': addr.get('address_1', ''),
            'city': addr.get('city', ''),
            'state': addr.get('state', ''),
            'zip': addr.get('zip_code', ''),
            'beds': item.get('beds', ''),
            'baths': item.get('baths', ''),
            'sqft': item.get('square_footage', ''),
            'rent': item.get('rent', ''),
            'total_monthly_rent': item.get('total_monthly_rent', ''),
            'status': item.get('status', ''),
            'available_on': item.get('available_on', ''),
            'property_type': item.get('property_type', ''),
            'is_on_special': item.get('is_on_special', False),
            'is_self_show_enabled': item.get('is_self_show_enabled', False),
            'has_virtual_tour': item.get('has_virtual_tour', False),
            'is_new_construction': item.get('is_new_construction', False),
            'community': item.get('community') or '',
            'market_name': item.get('market_name', ''),
            'lat': loc.get('latitude', ''),
            'lng': loc.get('longitude', ''),
            'link': 'https://www.invitationhomes.com/houses-for-rent/' + item.get('slug', ''),
            'metro_location': metro_location,
        })

    print(f'Offset {offset}/{total} — {len(data["properties"])} listings')
    offset += LIMIT

print(f'\nTotal records scraped: {len(records)}')

Offset 0/140 — 25 listings
Offset 25/140 — 25 listings
Offset 50/140 — 25 listings
Offset 75/140 — 25 listings
Offset 100/140 — 25 listings
Offset 125/140 — 15 listings

Total records scraped: 140


In [4]:
df = pd.DataFrame(records)

dupes = df.duplicated(subset='property_id', keep='first').sum()
print(f'Duplicate rows: {dupes}')
df = df.drop_duplicates(subset='property_id', keep='first')

df.info()
df.head()

Duplicate rows: 0
<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   property_id           140 non-null    str    
 1   street                140 non-null    str    
 2   city                  140 non-null    str    
 3   state                 140 non-null    str    
 4   zip                   140 non-null    str    
 5   beds                  140 non-null    int64  
 6   baths                 140 non-null    float64
 7   sqft                  140 non-null    int64  
 8   rent                  140 non-null    str    
 9   total_monthly_rent    140 non-null    str    
 10  status                140 non-null    str    
 11  available_on          140 non-null    str    
 12  property_type         140 non-null    str    
 13  is_on_special         140 non-null    bool   
 14  is_self_show_enabled  140 non-null    bool   
 15  has_virtual_tour

,property_id,street,city,state,zip,beds,baths,sqft,rent,total_monthly_rent,...,is_on_special,is_self_show_enabled,has_virtual_tour,is_new_construction,community,market_name,lat,lng,link,metro_location
0,ua000466,6325 Paddington Bend Dr,Houston,TX,77008,3,3.5,1820,2680.00,2739.90,...,True,True,False,False,"{'community_id': 'COM-0067', 'name': 'Palisade...",Houston,29.784806,-95.427218,https://www.invitationhomes.com/homes/6325-pad...,houston-texas
1,ua000417,6343 Paddington Bend Dr,Houston,TX,77008,3,2.5,1422,2399.00,2458.90,...,True,True,False,False,"{'community_id': 'COM-0067', 'name': 'Palisade...",Houston,29.784438,-95.427540,https://www.invitationhomes.com/homes/6343-pad...,houston-texas
2,11120087,8204 Sunberry Shadow Drive,Houston,TX,77016,4,2.5,2316,2449.00,2508.90,...,True,True,True,False,,Houston,29.863469,-95.279088,https://www.invitationhomes.com/homes/8204-sun...,houston-texas
3,11120085,8212 Sunberry Shadow Drive,Houston,TX,77016,4,2.5,2316,2449.00,2508.90,...,True,True,True,False,,Houston,29.863408,-95.278728,https://www.invitationhomes.com/homes/8212-sun...,houston-texas
4,11120084,8214 Sunberry Shadow Drive,Houston,TX,77016,3,2.5,1881,2249.00,2308.90,...,True,True,True,False,,Houston,29.863400,-95.278600,https://www.invitationhomes.com/homes/8214-sun...,houston-texas


In [5]:
df.to_csv('invitation_homes.csv', index=False)
print(f'Saved {len(df)} rows to invitation_homes.csv')

Saved 140 rows to invitation_homes.csv


In [6]:
conn = sqlite3.connect('invitation_homes.db')
df.to_sql('listings', conn, if_exists='replace', index=False)
print(f'Wrote {len(df)} rows to invitation_homes.db -> listings table')
conn.close()

DatabaseError: Execution failed